In [1]:
import numpy as np
import pandas as pd

In [ ]:
def extract_kinetic_features_per_rep(rep_df, rep_1_avg_ascent_vel=None):
    """
    Takes a dataframe containing exactly ONE repetition and calculates
    all 6 Kinetics & Velocity metrics + 2 Set Context features.
    
    Parameters:
    - rep_df: Dataframe of a single repetition slice
    - rep_1_avg_ascent_vel: The average ascent velocity of Rep 1 in this video set
                            (Used to calculate velocity loss from fatigue)
    """
    # Find the frame representing the absolute bottom of the squat
    bottom_idx = rep_df['hip_y_smooth'].idxmax()
    
    # Split the repetition into descent and ascent phases
    descent_df = rep_df.loc[:bottom_idx]
    ascent_df = rep_df.loc[bottom_idx:]

    # --- 1. Average Velocities (Your original metrics) ---
    # MediaPipe Y goes down, so descending yields positive velocity, ascending yields negative.
    # We take absolute values or map signs to make metrics intuitive.
    avg_descent_velocity_y = descent_df['hip_velocity_y'].mean()
    avg_ascent_velocity_y = abs(ascent_df['hip_velocity_y'].mean()) if len(ascent_df) > 0 else 0.0

    # --- 2. Acceleration and Jerk Spikes (Your original metrics) ---
    # Calculate derivatives locally across the frames of this specific rep
    hip_velocities = rep_df['hip_velocity_y'].values
    
    # Acceleration = derivative of velocity
    hip_accelerations = np.diff(hip_velocities, prepend=0)
    max_acceleration_y = np.max(np.abs(hip_accelerations)) if len(hip_accelerations) > 0 else 0.0
    
    # Jerk = derivative of acceleration
    hip_jerks = np.diff(hip_accelerations, prepend=0)
    max_jerk_y = np.max(np.abs(hip_jerks)) if len(hip_jerks) > 0 else 0.0

    # --- 3. Peak Ascent Velocity & Sticking Point Drop (Upgraded Metrics) ---
    # Good squats accelerate smoothly out of the hole. Bad squats stall out.
    if len(ascent_df) > 0:
        # Convert ascending velocities to positive values for simpler tracking
        ascent_speeds = np.abs(ascent_df['hip_velocity_y'].values)
        peak_ascent_velocity_y = np.max(ascent_speeds)
        
        # Locate when peak velocity happens during the drive up
        peak_ascent_idx = np.argmax(ascent_speeds)
        
        # Look for the minimum speed recorded AFTER hitting peak velocity but BEFORE finishing the rep
        post_peak_speeds = ascent_speeds[peak_ascent_idx:]
        min_ascent_velocity_post_peak = np.min(post_peak_speeds) if len(post_peak_speeds) > 0 else peak_ascent_velocity_y
        
        # Calculate the velocity drop (stalling out profile)
        sticking_point_velocity_drop = peak_ascent_velocity_y - min_ascent_velocity_post_peak
    else:
        peak_ascent_velocity_y = 0.0
        sticking_point_velocity_drop = 0.0

    # --- 4. Set Context & Fatigue (Final 2 Features) ---    
    # Velocity Loss Ratio relative to the first rep of the set
    if rep_1_avg_ascent_vel and rep_1_avg_ascent_vel > 0:
        velocity_loss_vs_rep_1 = avg_ascent_velocity_y / rep_1_avg_ascent_vel
    else:
        velocity_loss_vs_rep_1 = 1.0 # If it is Rep 1, there is no velocity loss yet

    return {
        'avg_descent_velocity_y': avg_descent_velocity_y,
        'avg_ascent_velocity_y': avg_ascent_velocity_y,
        'max_acceleration_y': max_acceleration_y,
        'max_jerk_y': max_jerk_y,
        'peak_ascent_velocity_y': peak_ascent_velocity_y,
        'sticking_point_velocity_drop': sticking_point_velocity_drop,
        'velocity_loss_vs_rep_1': velocity_loss_vs_rep_1
    }